# You ran the experiment. Now what happens to the model?

The holdout came back: a clean, randomized estimate of the lift. The panel model is still
fitted to observational data and still reports its own, wider, differently-centred number.
What usually happens next is that both numbers appear on the same slide and the reader picks
one — or worse, someone hard-codes the experiment's answer into the model as a fixed
coefficient, and the model's uncertainty about everything else silently stops being valid.

The prior route is the honest version: the experiment becomes a prior on the parameter it
actually informs, the fit runs again, and the result carries a ledger line for every step of
the translation.

The prior route (`derive_prior`) has two stages (decision D6.2):

1. **Combine.** Measurements of the *same* target estimand (equal content hashes) pool by
   inverse variance. A measurement of a different estimand is refused unless a correction factor
   from `calibrate.transfer` is supplied — and then a ledger line records it.
2. **Map to the amplitude.** The pooled `(mean, se)` is divided by the design factor
   `mean(contribution / beta)` and moment-matched to a lognormal or gamma prior on the
   amplitude parameter of the treatment's kernel.

The output is a `CalibratedSpec`: a *new* `SurfaceSpec` whose kernel carries
`amplitude_prior`, with every intermediate number and the ledger lines that license each step.
Only the amplitude prior changes — the curve's shape and scale priors are untouched, which is the
defining property of this route (the likelihood route in notebook 03 moves the shape too).

In [ ]:
import numpy as np

from axiom.calibrate import (
    CalibratedSpec, Measurement, PriorFamily, amplitude_prior, combine_measurements, derive_prior,
    design_factor,
)
from axiom.core import Intervention, Population, Posterior, Prior, TimeWindow, Unsupported
from axiom.estimands import Estimand, Level, Quantity, realize
from axiom.sim import surface_world
from axiom.surface import HillKernel, build, fit

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import BLUE, ORANGE, caption, density, dumbbell, mark_x

enable();  # every axiom result renders itself from here on

## A world, an observational panel, and a randomized experiment

The world is a one-treatment Hill surface with a shared intercept. The observational panel is
noisy (`noise_sd = 0.6` on a response whose amplitude is `1.5`), so the posterior on the amplitude
from the panel alone is wide. The "experiment" is simulated from the world's truth: the estimand
is the per-unit cumulative contrast between dose 2 and dose 0 over the full horizon, and the
experiment reports it with a standard error of 5% of the truth.

In [ ]:
TRUTH = {"beta_a": 1.5, "k_a": 1.0, "s_a": 2.0}
world = surface_world(
    n_units=4, n_periods=12, treatments=("a",), kernels=HillKernel(reference_dose=1.0),
    intercept="shared", truth=TRUTH, noise_sd=0.6, seed=7,
)
spec = world.spec
lift = Estimand(
    name="lift_2_vs_0",
    quantity=Quantity(kind="contrast"),
    treatment=spec.treatment("a"),
    intervention=Intervention(doses={"a": 2.0}),
    reference=Intervention(doses={"a": 0.0}),
    outcome=spec.outcome,
    population=Population(name="panel_units"),
    window=TimeWindow(start=0, stop=world.n_periods, basis="cumulative"),
    level=Level(unit="individual"),
    dimension=spec.outcome_dimension,
)
truth_contrast = float(np.mean(np.sum(world.forward({"a": 2.0}) - world.forward({"a": 0.0}), axis=1)))
rng = np.random.default_rng(1)
se_exp = 0.05 * truth_contrast
experiment = Measurement(
    estimand=lift, estimate=truth_contrast + rng.normal(0.0, se_exp), se=se_exp,
    method="randomized_holdout", n_units=40, n_periods=12, source="holdout-2026Q1",
)
print(f"true contrast {truth_contrast:.4f}; experiment reads {experiment.estimate:.4f} ± {experiment.se:.4f}")

## The uncalibrated fit supplies the design factor

The design factor needs paired draws of the amplitude and of the realized target. The
uncalibrated Laplace fit gives both: `beta_a` from the posterior, and the contrast from
`estimands.realize` on the same draws (`keep_draws=True`), through the one `forward()`.

In [ ]:
plain = fit(spec, world.panel, backend="laplace", draws=500, seed=0)
assert isinstance(plain.posterior, Posterior)
beta_draws = plain.posterior.flat("beta_a")
out = realize(lift, plain, assume_identified=True, keep_draws=True, seed=0)
contribution_draws = np.asarray(out.draws)
f = design_factor(beta_draws, contribution_draws)
print(f"uncalibrated beta_a: {beta_draws.mean():.3f} ± {beta_draws.std():.3f}  (truth {TRUTH['beta_a']})")
print(f"design factor mean(contrast / beta_a) = {f:.3f}  (the contrast is ~{f:.1f} outcome units per unit amplitude)")

## Stage 1 on its own: `combine_measurements`

`combine_measurements` pools on the first measurement's estimand by default and returns the
pooled `(mean, se)`, the ledger lines it wrote, and a status per source. A second measurement of
the *same* estimand is `identified`; one of a different estimand without a correction factor is a
typed `Unsupported` naming the source, never a silently pooled number.

In [ ]:
second = Measurement(estimand=lift, estimate=truth_contrast * 1.03, se=2 * se_exp, source="holdout-2025Q3")
combined = combine_measurements([experiment, second])
assert not isinstance(combined, Unsupported)
mean, se, lines, statuses = combined
print(f"pooled {mean:.4f} ± {se:.4f}; statuses {statuses}")
print(lines[0].kind, "|", lines[0].assumption.name, "| counterfactual", lines[0].detail["counterfactual"][:8], "-> value", lines[0].detail["value"][:8])

other = Estimand.model_validate({**lift.model_dump(), "name": "lift_short", "window": TimeWindow(start=0, stop=4, basis="cumulative").model_dump()})
refused = combine_measurements([experiment, Measurement(estimand=other, estimate=1.0, se=0.1, source="short-window")])
print(type(refused).__name__, "->", refused.missing)

## Stage 2: `derive_prior` → `CalibratedSpec`

The result records every number along the way: the pooled target, the design factor, the implied
amplitude moments, the prior, and three ledger lines (`evidence:combine`, `prior:design_factor`,
`prior:moment_match`), each with a typed assumption and a counterfactual.

In [ ]:
family: PriorFamily = "lognormal"
cal = derive_prior([experiment, second], spec, "a", beta_draws=beta_draws,
                   contribution_draws=contribution_draws, family=family)
assert isinstance(cal, CalibratedSpec)
print(f"target {cal.target_mean:.4f} ± {cal.target_se:.4f} | design factor {cal.design_factor:.3f}")
print(f"amplitude {cal.parameter}: {cal.amplitude_mean:.4f} ± {cal.amplitude_sd:.4f} -> {cal.prior}")
print("plan statuses:", cal.plan_statuses)
table(
    [
        [line.kind, line.assumption.name, f"{float(line.detail['counterfactual']):.4g}",
         f"{float(line.detail['value']):.4g}"]
        for line in cal.ledger_lines
    ],
    headers=("kind", "assumption", "counterfactual", "value"),
)

## Only the amplitude prior changed

`cal.spec` is the input spec with one kernel's `amplitude_prior` set; `surface.build` uses it in
place of the default `halfnormal(amplitude_scale)`. The shape and scale parameters keep their
priors — compare the built models parameter by parameter.

In [ ]:
print("kernel before:", spec.kernel_of("a"))
print("kernel after: ", cal.spec.kernel_of("a"))
before = {p.name: p.prior for p in build(spec).parameters}
after = {p.name: p.prior for p in build(cal.spec).parameters}
table(
    [
        [name, "CHANGED" if before[name] != after[name] else "same", after[name].family]
        for name in before
    ],
    headers=("parameter", "prior", "family after"),
)
print("spec hash changed:", spec.content_hash() != cal.spec.content_hash())

## Fitting the calibrated spec moves the amplitude posterior

Refit with the calibrated prior. The amplitude posterior tightens and moves toward the truth. The
shape and scale *priors* are untouched; their posteriors can still shift, but only indirectly,
through their posterior correlation with the amplitude on this small noisy panel. That indirect
pull is all the prior route allows — the likelihood route (notebook 03) constrains an expression
that depends on the shape directly.

In [ ]:
calibrated = fit(cal.spec, world.panel, backend="laplace", draws=500, seed=0)
assert isinstance(calibrated.posterior, Posterior)
rows = []
for name in ("beta_a", "s_a", "k_a"):
    a, b = plain.posterior.flat(name), calibrated.posterior.flat(name)
    rows.append(
        [name, f"{a.mean():.3f} ± {a.std():.3f}", f"{b.mean():.3f} ± {b.std():.3f}", TRUTH[name]]
    )
table(rows, headers=("parameter", "plain", "calibrated", "truth"))
beta_plain, beta_cal = plain.posterior.flat("beta_a"), calibrated.posterior.flat("beta_a")
print("amplitude sd shrank:", beta_cal.std() < beta_plain.std(),
      "| nearer the truth:", abs(beta_cal.mean() - 1.5) < abs(beta_plain.mean() - 1.5))

In [ ]:
fig = density(
    {"panel alone": beta_plain, "panel + the experiment": beta_cal},
    colors=(BLUE, ORANGE),
    title="What the experiment did to the amplitude",
    subtitle="posterior for beta_a before and after the randomized measurement became a prior",
    x_title="beta_a",
)
mark_x(fig, TRUTH["beta_a"], text="truth")
caption(fig, f"The observational panel is noisy enough that its amplitude posterior is wide "
             f"and off-centre. One randomized measurement, translated through the design "
             f"factor, narrows it from ±{beta_plain.std():.2f} to ±{beta_cal.std():.2f} and "
             f"halves its distance from the truth — without anybody overwriting a coefficient.")

In [ ]:
names = ("beta_a", "s_a", "k_a")
fig = dumbbell(
    list(names),
    [plain.posterior.flat(n).std() for n in names],
    [calibrated.posterior.flat(n).std() for n in names],
    before_label="panel alone", after_label="after calibration",
    title="…and what it did to everything else",
    subtitle="posterior standard deviation per parameter, prior route",
    x_title="posterior sd",
)
caption(fig, "The amplitude's sd falls by a factor of seven. The shape tightens slightly and "
             "the scale not at all — whatever they gain arrives only through their posterior "
             "correlation with the amplitude. That is the defining limit of this route, and "
             "the likelihood route in notebook 03 is what lifts it.")

## `amplitude_prior` on a kernel directly

Every kernel `Spec` accepts an explicit `amplitude_prior`; it must have positive support
(`lognormal`, `gamma`, `halfnormal`) and may not reference other parameters. A `normal` is
refused at construction.

In [ ]:
prior = amplitude_prior(1.5, 0.2, "gamma")
print(HillKernel(reference_dose=1.0, amplitude_prior=prior))
try:
    HillKernel(reference_dose=1.0, amplitude_prior=Prior(family="normal", hyper={"mu": 1.0, "sigma": 1.0}))
except ValueError as e:
    print("ValueError:", next(line.strip() for line in str(e).splitlines() if "amplitude_prior" in line))

## What this bought you

A randomized result folded into the model as a prior on the parameter it informs, with the
design factor that translated it, the family it was matched to, and a ledger line per step —
instead of two numbers on a slide and a reader choosing between them.